# Inspect panel j variants: joint SpatialLeiden on Kuppe 2022 (27 slides)

Every run is **one joint clustering of all 27 Visium slides** (88,704 spots). The expression-type graph is 15-NN on a **Harmony-integrated** embedding, and the spatial graph is a hexagonal grid built per slide.

| Layer | Embedding | Harmony covariates |
|---|---|---|
| RNA | `results/kuppe_2022/exp1_harmony_covariates/embeddings.npz`, key `authors` | patient + sample + batch |
| TF (CollecTRI) | `results/kuppe_2022/tf_harmony/tf_embeddings.npz`, key `X_harmony` | patient + sample + batch |
| CCC | colleague's `CCC_all27_TFmatched_Harmony.h5ad`, obsm `X_pca_harmony_TFmatched` | patient + sample |

**Two kinds of run:**

- **Fixed K = 9** (experiments 2 and 3): the resolution is searched at every spatial weight so that every column has exactly 9 domains. Seeds 0–4; the panels show seed 0.
- **Free K** (experiments 4 and 5): the expression resolution is chosen so that weight 0 gives K = 9, and the spatial resolution is fixed at 1.0. Then there is one run per weight, so K can change with the weight. Seed 0.

**Domain colours:** D1, D2, … are numbered by size at weight 0 over all spots. At the other weights, each domain takes the number of the weight-0 domain it overlaps most (Hungarian matching). With free K, extra domains get new numbers.

Run from `Derived_Modalities/` with the `spatialleiden` env.

In [ ]:
import os, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree
from IPython.display import Image, display

ROOT = Path.cwd()
while not (ROOT / "01_data_preprocess").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
DATA = Path("data/kuppe_2022")
RES = Path("results/kuppe_2022")
FIGDIR = Path("figures/kuppe_fig1h/inspect")
FIGDIR.mkdir(parents=True, exist_ok=True)
PY = sys.executable
print("working in", ROOT)

## 1. Runs

In [ ]:
# name -> (labels file, panel-j title text, free K?)
RUNS = {
    "RNA (K = 9)":                (RES / "exp2_joint_harmony/exp2_labels.csv.gz", "RNA", False),
    "TF (K = 9)":                 (RES / "exp2_joint_harmony_tf/exp2_labels.csv.gz", "TF activity", False),
    "RNA + TF + space (K = 9)":   (RES / "exp3_proto_rna_tf/exp2_labels.csv.gz", "RNA + TF (0.5)", False),
    "RNA + CCC + space (K = 9)":  (RES / "exp3_joint_multimodal_rna_ccc/exp2_labels.csv.gz", "RNA + CCC (0.5)", False),
    "RNA (free K)":               (RES / "exp4_unfixed_k_rna/exp2_labels.csv.gz", "RNA", True),
    "TF (free K)":                (RES / "exp4_unfixed_k_tf/exp2_labels.csv.gz", "TF activity", True),
    "CCC (free K)":               (RES / "exp4_unfixed_k_ccc/exp2_labels.csv.gz", "CCC", True),
    "RNA + CCC (free K)":         (RES / "exp5_unfixed_k_rna_ccc/exp2_labels.csv.gz", "RNA + CCC (0.5)", True),
    "RNA + CCC + TF (free K)":    (RES / "exp5_unfixed_k_rna_ccc_tf/exp2_labels.csv.gz", "RNA + CCC + TF (0.5 each)", True),
}
WEIGHTS = [0, 0.5, 1.0, 1.5]
FIG_ROWS = [("ACH003", "Ctrl P7"), ("10X0026", "BZ P3"), ("10X0017", "IZ P3"), ("ACH005", "FZ P14")]
avail = {k: v for k, v in RUNS.items() if v[0].exists()}
print("available:", list(avail)); print("missing (not run / still running):", [k for k in RUNS if k not in avail])

## 2. Number of domains per weight (overall and on the four Fig. 1 slides)

In [ ]:
def k_table(path):
    lab = pd.read_csv(path, index_col=0, dtype=str)
    smp = lab.index.str.split("_", n=1).str[0]
    rows = []
    for w in WEIGHTS:
        c = f"r{w}_s0"
        if c not in lab: continue
        r = {"weight": w, "K (all 27 slides)": lab[c].nunique()}
        for s, name in FIG_ROWS:
            r[name] = lab.loc[smp == s, c].nunique()
        # domains covering >= 1 % of all spots: the free-K runs add many tiny islands
        r["K >= 1 % of spots"] = int((lab[c].value_counts(normalize=True) >= 0.01).sum())
        rows.append(r)
    return pd.DataFrame(rows).set_index("weight")

for name, (path, _, _) in avail.items():
    print(name); display(k_table(path))

## 3. Fig. 1h–j per run

Panels h and i are the same for every run; j is the run. The figures are made with `01_data_preprocess/kuppe_fig1/fig1hij.py` and saved in `figures/kuppe_fig1h/inspect/`. Set `REDRAW = True` to re-render.

In [ ]:
REDRAW = False
def slug(name): return "".join(ch if ch.isalnum() else "_" for ch in name).strip("_").lower()

def fig1(name):
    path, layer, free = avail[name]
    out = FIGDIR / f"fig1hij_{slug(name)}.png"
    if REDRAW or not out.exists():
        env = {**os.environ, "FIG_SL_RATIOS": ",".join(map(str, WEIGHTS))}
        if free: env["FIG_FREE_K"] = "1"
        subprocess.run([PY, "-W", "ignore", "01_data_preprocess/kuppe_fig1/fig1hij.py", str(DATA), str(FIGDIR),
                        str(path), out.stem, layer], env=env, check=True, capture_output=True)
    return out

for name in avail:
    print(name); display(Image(filename=str(fig1(name)), width=1400))

## 4. Fig. 2b per run (authors' niches on P1 and P9, then SpatialLeiden)

In [ ]:
def fig2(name):
    path, layer, free = avail[name]
    out = FIGDIR / f"fig2b_{slug(name)}.png"
    if REDRAW or not out.exists():
        env = {**os.environ, "FIG_SL_RATIOS": ",".join(map(str, WEIGHTS))}
        r = subprocess.run([PY, "-W", "ignore", "01_data_preprocess/kuppe_fig2/fig2b_spatialleiden.py", str(DATA),
                            str(FIGDIR), str(path), out.stem, layer], env=env, capture_output=True, text=True)
        if r.returncode:  # fig2b script assumes the same K at every weight
            print(f"  {name}: Fig. 2b not drawn ({r.stderr.strip().splitlines()[-1]})"); return None
    return out

for name in avail:
    p = fig2(name)
    if p: print(name); display(Image(filename=str(p), width=1200))

## 5. Interactive: any slide, any runs, side by side

`compare(slide, runs, weights)` draws one row per run. Colours are matched within each run (to its weight 0), not across runs. Slides are Visium run IDs: Ctrl P7 `ACH003`, BZ P3 `10X0026`, IZ P3 `10X0017`, FZ P14 `ACH005`, P1 `10X001`, P9 `ACH0012`, P9 rep2 `ACH0013`, and so on.

In [ ]:
PAL = np.array(["#E69F00","#56B4E9","#009E73","#F0E442","#0072B2","#D55E00","#CC79A7","#FFFFFF","#8C6BB1",
                "#A6761D","#66C2A5","#E7298A","#B3DE69","#FB8072","#80B1D3","#BEBADA","#FFED6F","#1B9E77",
                "#7570B3","#D9D9D9","#BC80BD","#CCEBC5","#FDB462"] + ["#999999"] * 60)
_cache = {}
def labels(name):
    if name not in _cache: _cache[name] = pd.read_csv(avail[name][0], index_col=0, dtype=str)
    return _cache[name]

def matched(lab, cols):
    ref = lab[cols[0]]; num = {c: i for i, c in enumerate(ref.value_counts().index)}
    out = {cols[0]: ref.map(num).to_numpy()}; nxt = len(num)
    for c in cols[1:]:
        tab = pd.crosstab(lab[c], ref.map(num)); r_, c_ = linear_sum_assignment(-tab.to_numpy())
        m = {tab.index[a]: tab.columns[b] for a, b in zip(r_, c_)}
        for extra in lab[c][~lab[c].isin(m)].value_counts().index:
            m[extra] = nxt; nxt += 1
        out[c] = lab[c].map(m).to_numpy()
    return out

def compare(slide="10X0017", runs=None, weights=WEIGHTS, size=3):
    runs = runs or list(avail)
    sp = pd.read_csv(DATA / "visium_tables" / f"{slide}_spots.csv")
    x, y = sp.pxl_col_fullres.to_numpy(float), sp.pxl_row_fullres.to_numpy(float)
    p = np.median(cKDTree(np.c_[x, y]).query(np.c_[x, y], k=2)[0][:, 1])
    fig, ax = plt.subplots(len(runs), len(weights), figsize=(size * len(weights), size * len(runs)), squeeze=False)
    for i, name in enumerate(runs):
        lab = labels(name)
        cols = [f"r{w}_s0" for w in weights if f"r{w}_s0" in lab]
        m = matched(lab, cols)
        idx = pd.Index(lab.index).get_indexer(slide + "_" + sp.barcode)
        for j, c in enumerate(cols):
            a = ax[i, j]; a.set_facecolor("black")
            a.add_collection(EllipseCollection(np.full(len(x), p), np.full(len(x), p), np.zeros(len(x)), units="xy",
                             offsets=np.c_[x, y], offset_transform=a.transData, facecolors=PAL[m[c][idx]], linewidths=0))
            a.set_xlim(x.min() - p, x.max() + p); a.set_ylim(y.max() + p, y.min() - p); a.set_aspect("equal")
            a.set_xticks([]); a.set_yticks([])
            a.set_title(f"weight {c[1:].split('_')[0]}  (K on slide {len(set(m[c][idx]))})", fontsize=8)
            if j == 0: a.set_ylabel(name, fontsize=8)
    fig.suptitle(f"slide {slide}"); plt.tight_layout(); plt.show()

compare("10X0017")

In [ ]:
compare("ACH0013", runs=[r for r in avail if "free K" in r])   # P9 rep2: the vessel slide

In [ ]:
# agreement between runs at the same weight (ARI over all 88,704 spots)
from sklearn.metrics import adjusted_rand_score as ari
def ari_matrix(w=1.0):
    names = [n for n in avail if f"r{w}_s0" in labels(n)]
    M = pd.DataFrame(index=names, columns=names, dtype=float)
    for a in names:
        for b in names:
            M.loc[a, b] = ari(labels(a)[f"r{w}_s0"], labels(b).loc[labels(a).index, f"r{w}_s0"])
    return M.round(2)
display(ari_matrix(0)); display(ari_matrix(1.0))